In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import re
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

BASE = Path(r"d:/sugarcare_backend")
DATA_DIR = BASE / "data" / "your_datasets"
CHATBOT_DIR = DATA_DIR / "chatbot"
MODELS_DIR = BASE / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("SugarCare Chatbot Training")
print("=" * 50)

In [ ]:
def read_csv_file(p: Path, max_rows=None):
    """Read CSV file with multiple encoding attempts."""
    for encoding in [None, 'utf-8', 'latin-1', 'cp1252']:
        try:
            kwargs = {'engine': 'python'}
            if encoding:
                kwargs['encoding'] = encoding
            if max_rows:
                kwargs['nrows'] = max_rows
            return pd.read_csv(p, **kwargs)
        except Exception:
            continue
    return None

def normalize_text(s):
    """Clean and normalize text."""
    if pd.isna(s):
        return ""
    s = str(s).strip()
    s = re.sub(r'\s+', ' ', s)
    return s

def is_valid_qa(question, answer):
    """Check if Q/A pair is valid (not numeric garbage)."""
    q = str(question).strip()
    a = str(answer).strip()
    
    if len(q) < 10 or len(a) < 20:
        return False
    
    q_alpha = sum(c.isalpha() for c in q)
    a_alpha = sum(c.isalpha() for c in a)
    if q_alpha < len(q) * 0.5 or a_alpha < len(a) * 0.5:
        return False
    
    garbage_patterns = ['0.0', '1.0', '2.0', 'nan', 'none', 'null']
    if any(p in a.lower() for p in garbage_patterns):
        return False
    
    return True

In [ ]:
manual_qa = [
    ("what is normal blood sugar level", "Normal fasting blood sugar is 70-99 mg/dL. After meals (2 hours), normal is under 140 mg/dL. Random blood sugar under 140 mg/dL is also considered normal."),
    ("what is the range of blood sugar", "Blood sugar ranges: Normal fasting is 70-99 mg/dL. Prediabetes is 100-125 mg/dL fasting. Diabetes is 126 mg/dL or higher fasting. After meals, under 140 is normal, 140-199 is prediabetes, and 200+ indicates diabetes."),
    ("what is prediabetes", "Prediabetes means your blood sugar is higher than normal but not high enough for diabetes. Fasting blood sugar 100-125 mg/dL or HbA1c 5.7-6.4% indicates prediabetes. With lifestyle changes, you can prevent or delay type 2 diabetes."),
    ("what is diabetes", "Diabetes is a condition where your body cannot properly process blood sugar (glucose). Type 1 diabetes means your body doesn't make insulin. Type 2 diabetes means your body doesn't use insulin well. Both cause high blood sugar levels."),
    ("what is normal hba1c", "Normal HbA1c is below 5.7%. Prediabetes is 5.7% to 6.4%. Diabetes is diagnosed at 6.5% or higher. For people with diabetes, the target is usually below 7%, but your doctor may set a different goal."),
    ("tell me the range of healthy prediabetes and diabetes", "Healthy/Normal: Fasting blood sugar 70-99 mg/dL, HbA1c under 5.7%. Prediabetes: Fasting 100-125 mg/dL, HbA1c 5.7-6.4%. Diabetes: Fasting 126+ mg/dL, HbA1c 6.5% or higher."),
    ("how to maintain sugar level", "To maintain healthy blood sugar: 1) Eat balanced meals with vegetables, whole grains, and lean protein. 2) Limit sugary foods and refined carbs. 3) Exercise 30 minutes daily. 4) Stay hydrated. 5) Monitor blood sugar regularly. 6) Take medications as prescribed. 7) Get enough sleep."),
    ("how to control blood sugar", "Control blood sugar by: eating smaller portions, choosing low-glycemic foods, exercising regularly (walking helps!), reducing stress, getting adequate sleep, staying hydrated, and taking medications as directed. Regular monitoring helps track your progress."),
    ("how to reduce sugar level", "To reduce high blood sugar: 1) Drink plenty of water. 2) Take a 15-30 minute walk. 3) Eat fiber-rich foods. 4) Avoid sugary drinks and snacks. 5) Take your medication as prescribed. If consistently high, consult your doctor."),
    ("how to manage diabetes", "Manage diabetes by: monitoring blood sugar regularly, eating healthy balanced meals, exercising daily, taking medications as prescribed, maintaining healthy weight, managing stress, getting regular checkups, and caring for your feet and eyes."),
    ("what to do when sugar is low", "When blood sugar is low (under 70 mg/dL): 1) Eat 15-20 grams of fast-acting carbs like glucose tablets, 4 oz juice, or regular soda. 2) Wait 15 minutes and recheck. 3) Repeat if still low. 4) Once normal, eat a snack with protein. Seek help if symptoms don't improve."),
    ("my sugar is dropping", "If your sugar is dropping frequently: Don't skip meals, eat regular snacks between meals, check blood sugar before exercise, talk to your doctor about medication adjustments. Signs of low sugar: shakiness, sweating, confusion, dizziness, hunger."),
    ("symptoms of low blood sugar", "Low blood sugar (hypoglycemia) symptoms include: shakiness, sweating, fast heartbeat, dizziness, hunger, irritability, confusion, blurred vision, and weakness. If severe, it can cause seizures or loss of consciousness. Treat immediately with fast-acting carbs."),
    ("what to do when sugar is high", "When blood sugar is high: 1) Drink plenty of water. 2) Take a walk if you feel well. 3) Avoid sugary foods. 4) Take medication as prescribed. 5) Check for ketones if very high. Seek medical help if: above 300 mg/dL, have ketones, feel nauseous or confused."),
    ("why is my sugar high", "High blood sugar can be caused by: eating too many carbs, missing medication, being sick or stressed, not exercising enough, certain medications, or dehydration. Regular monitoring and lifestyle management help prevent spikes."),
    ("symptoms of high blood sugar", "High blood sugar (hyperglycemia) symptoms include: increased thirst, frequent urination, blurred vision, fatigue, headache, and slow healing wounds. Very high levels can cause nausea, vomiting, confusion, and fruity-smelling breath."),
    ("what are symptoms of diabetes", "Common diabetes symptoms: increased thirst, frequent urination, extreme hunger, unexplained weight loss, fatigue, blurred vision, slow-healing sores, frequent infections, and numbness/tingling in hands or feet. If you have these, see a doctor."),
    ("how do i know if i have diabetes", "Signs you might have diabetes: frequent urination especially at night, extreme thirst, unexplained weight loss, fatigue, blurred vision, slow healing cuts. Only a blood test can confirm diabetes - see your doctor for fasting glucose or HbA1c testing."),
    ("what should diabetic eat", "Good foods for diabetics: non-starchy vegetables (broccoli, spinach, peppers), whole grains (brown rice, oats), lean proteins (chicken, fish, beans), healthy fats (nuts, olive oil), and low-glycemic fruits (berries, apples, oranges). Limit white bread, sugary drinks, and sweets."),
    ("what foods to avoid diabetes", "Foods to avoid or limit: sugary drinks (soda, juice), white bread and rice, fried foods, processed snacks, candy and sweets, high-fat meats, full-fat dairy. Choose whole foods and watch portion sizes instead."),
    ("can diabetic eat fruit", "Yes, diabetics can eat fruit! Choose low-glycemic fruits like berries, apples, oranges, and pears. Eat in moderation, prefer whole fruit over juice, and pair with protein or healthy fat. Limit high-sugar fruits like mangoes, grapes, and bananas."),
    ("can diabetic eat rice", "Diabetics can eat rice in moderation. Choose brown rice, quinoa, or cauliflower rice over white rice. Control portion size (1/2 cup cooked), pair with vegetables and protein, and monitor how it affects your blood sugar."),
    ("exercise for diabetes", "Best exercises for diabetes: walking (30 min daily), swimming, cycling, light weight training, yoga, and dancing. Exercise lowers blood sugar, improves insulin sensitivity, and reduces stress. Start slowly and check blood sugar before and after."),
    ("how does exercise help diabetes", "Exercise helps diabetes by: lowering blood sugar immediately, improving insulin sensitivity for 24-72 hours, helping with weight management, reducing stress, lowering blood pressure and cholesterol, and improving overall energy and mood."),
    ("best time to exercise diabetes", "Good times to exercise: 1-3 hours after meals when blood sugar is slightly higher. A short walk after meals can help lower post-meal spikes. Avoid exercise if fasting blood sugar is above 250 mg/dL or you have ketones."),
    ("what is metformin", "Metformin is the most common medication for type 2 diabetes. It helps lower blood sugar by reducing glucose production in the liver and improving insulin sensitivity. Take with food to reduce stomach upset. Common side effects include nausea and diarrhea."),
    ("what is insulin", "Insulin is a hormone that helps glucose enter your cells for energy. People with type 1 diabetes need insulin injections. Some type 2 diabetics also need insulin. Types include rapid-acting, short-acting, intermediate, and long-acting insulin."),
    ("diabetes complications", "Diabetes complications include: heart disease, stroke, kidney damage (nephropathy), eye damage (retinopathy), nerve damage (neuropathy), foot problems, skin conditions, and hearing impairment. Good blood sugar control helps prevent these."),
    ("how to prevent diabetes complications", "Prevent complications by: keeping blood sugar in target range, controlling blood pressure and cholesterol, not smoking, exercising regularly, eating healthy, taking medications as prescribed, getting regular eye and foot exams, and managing stress."),
    ("how are you", "I'm doing well, thank you for asking! I'm here to help you with any questions about diabetes, blood sugar management, diet, exercise, or medications. How can I assist you today?"),
    ("what can you help with", "I can help you with: understanding blood sugar levels and ranges, tips for managing diabetes, diet and food choices, exercise recommendations, medication information, recognizing symptoms, and answering general diabetes questions."),
    ("hello", "Hello! I'm your diabetes assistant. I can help you understand blood sugar levels, give diet and exercise tips, and answer questions about diabetes management. What would you like to know?"),
    ("hi", "Hi there! I'm here to help with your diabetes and blood sugar questions. Feel free to ask about blood sugar ranges, diet tips, exercise, medications, or anything related to diabetes management."),
]

synonyms = {
    'sugar': ['blood sugar', 'glucose', 'blood glucose'],
    'sugar level': ['blood sugar level', 'glucose level', 'blood glucose level'],
    'diabetes': ['diabetic', 'diabetics'],
    'maintain': ['control', 'manage', 'keep stable'],
    'reduce': ['lower', 'decrease', 'bring down'],
}

expanded_qa = []
for q, a in manual_qa:
    expanded_qa.append((q, a))
    for word, syns in synonyms.items():
        if word in q.lower():
            for syn in syns:
                new_q = q.lower().replace(word, syn)
                if new_q != q.lower():
                    expanded_qa.append((new_q, a))

print(f"Manual Q/A pairs: {len(manual_qa)}")
print(f"After synonym expansion: {len(expanded_qa)}")

In [ ]:
qa_rows = []

manual_df = pd.DataFrame(expanded_qa, columns=['question', 'answer'])
qa_rows.append(manual_df)
qa_rows.append(manual_df.copy())
qa_rows.append(manual_df.copy())

if CHATBOT_DIR.exists():
    chatbot_files = list(CHATBOT_DIR.glob('*.csv'))
    print(f"\nFound {len(chatbot_files)} files in chatbot folder")
    
    for p in chatbot_files:
        print(f"Processing: {p.name}")
        df = read_csv_file(p)
        if df is None or df.empty:
            print(f"  Skipped (empty or unreadable)")
            continue
        
        q_col = a_col = None
        for col in df.columns:
            col_lower = str(col).lower()
            if q_col is None and any(k in col_lower for k in ['question', 'query', 'prompt', 'input', 'user']):
                q_col = col
            if a_col is None and any(k in col_lower for k in ['answer', 'response', 'reply', 'output', 'bot']):
                a_col = col
        
        if q_col is None or a_col is None:
            print(f"  Skipped (no Q/A columns found)")
            continue
        
        sub = df[[q_col, a_col]].dropna()
        sub.columns = ['question', 'answer']
        
        valid_pairs = []
        for _, row in sub.iterrows():
            if is_valid_qa(row['question'], row['answer']):
                valid_pairs.append({
                    'question': normalize_text(row['question']),
                    'answer': normalize_text(row['answer'])
                })
        
        if valid_pairs:
            valid_df = pd.DataFrame(valid_pairs)
            qa_rows.append(valid_df)
            print(f"  Added {len(valid_df)} valid Q/A pairs")
        else:
            print(f"  Skipped (no valid Q/A pairs)")

diabetes_qa_path = DATA_DIR / 'diabetes_qa_dataset.csv'
if diabetes_qa_path.exists():
    print(f"\nProcessing: diabetes_qa_dataset.csv")
    df = read_csv_file(diabetes_qa_path)
    if df is not None and not df.empty:
        q_col = a_col = None
        for col in df.columns:
            col_lower = str(col).lower()
            if 'question' in col_lower:
                q_col = col
            if 'answer' in col_lower:
                a_col = col
        
        if q_col and a_col:
            sub = df[[q_col, a_col]].dropna()
            sub.columns = ['question', 'answer']
            valid_pairs = []
            for _, row in sub.iterrows():
                if is_valid_qa(row['question'], row['answer']):
                    valid_pairs.append({
                        'question': normalize_text(row['question']),
                        'answer': normalize_text(row['answer'])
                    })
            if valid_pairs:
                valid_df = pd.DataFrame(valid_pairs)
                qa_rows.append(valid_df)
                qa_rows.append(valid_df.copy())
                print(f"  Added {len(valid_df)} valid Q/A pairs (2x weight)")

In [ ]:
if qa_rows:
    qa_df = pd.concat(qa_rows, ignore_index=True)
else:
    raise RuntimeError("No Q/A data found!")

qa_df = qa_df.dropna()
qa_df['question'] = qa_df['question'].astype(str).apply(normalize_text)
qa_df['answer'] = qa_df['answer'].astype(str).apply(normalize_text)

qa_df = qa_df[(qa_df['question'] != '') & (qa_df['answer'] != '')]
qa_df = qa_df[qa_df['question'].str.len() >= 5]
qa_df = qa_df[qa_df['answer'].str.len() >= 20]

qa_df = qa_df.drop_duplicates(subset=['question', 'answer']).reset_index(drop=True)

print(f"\nFinal dataset: {len(qa_df)} Q/A pairs")
print(f"\nSample Q/A pairs:")
for i in range(min(5, len(qa_df))):
    print(f"\nQ: {qa_df.iloc[i]['question'][:80]}...")
    print(f"A: {qa_df.iloc[i]['answer'][:100]}...")

In [ ]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    ngram_range=(1, 3),
    sublinear_tf=True,
    max_df=0.9,
    min_df=1,
    max_features=50000,
)

X = vectorizer.fit_transform(qa_df['question'].tolist())
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"Feature matrix shape: {X.shape}")

print("\nBuilding retriever...")
retriever = NearestNeighbors(
    n_neighbors=10,
    metric='cosine',
    algorithm='brute'
)
retriever.fit(X)
print("Retriever ready!")

In [ ]:
test_questions = [
    "how to maintain sugar level",
    "what is normal blood sugar",
    "what are symptoms of diabetes",
    "what should diabetic eat",
    "my sugar is high what to do",
    "range of healthy prediabetes diabetes",
]

print("Testing retriever:")
print("=" * 60)

for q in test_questions:
    query_vec = vectorizer.transform([q])
    distances, indices = retriever.kneighbors(query_vec, n_neighbors=3)
    
    print(f"\nQ: {q}")
    for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        sim = 1 - dist
        ans = qa_df.iloc[idx]['answer'][:100]
        print(f"  {i+1}. (sim={sim:.2f}) {ans}...")

In [ ]:
export = {
    'vectorizer': vectorizer,
    'retriever': retriever,
    'questions': qa_df['question'].tolist(),
    'answers': qa_df['answer'].tolist(),
    'score_threshold': 0.25,
    'model': None,
    'scaler': None,
}

out_path = MODELS_DIR / 'chatbot_responses_trained.pkl'
joblib.dump(export, out_path)

print(f"\n" + "=" * 60)
print("MODEL SAVED SUCCESSFULLY!")
print(f"=" * 60)
print(f"Path: {out_path}")
print(f"Total Q/A pairs: {len(qa_df)}")
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"Similarity threshold: 0.25")
print(f"\nThe chatbot is now ready to use!")